In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import datetime
# === 1. PERSIAPAN DATASET & AUGMENTASI ===
train_dir = "seg_train/seg_train"
val_dir = "seg_test/seg_test"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)
val_generator = val_datagen.flow_from_directory(
    val_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')
])

# Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Training model
model.fit(train_generator, validation_data=val_generator, epochs=10)

# Simpan model
model.save('cnn_model.h5')

# Load label kelas
class_labels = list(train_generator.class_indices.keys())

In [ ]:
def analyze_light_spectrum(image):
    """Menganalisis spektrum cahaya berdasarkan luminance."""
    # Konversi ke YUV untuk mendapatkan luminance (Y channel)
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    luminance = yuv[:, :, 0]
    avg_luminance = np.mean(luminance)
    
    # Menampilkan histogram luminance
    plt.figure(figsize=(7, 5))
    
   # Histogram Luminance (Y channel)
    hist_y = cv2.calcHist([luminance], [0], None, [300], [0, 300])
    plt.plot(hist_y, color='black')
    plt.title(f"Luminance Histogram\nBrightness: {avg_luminance:.2f}")
    plt.xlabel("Intensitas Luminance")
    plt.ylabel("Jumlah Piksel")
    
    plt.subplots_adjust(bottom=0.2)

    # Menampilkan waktu
    current_time = datetime.datetime.now().strftime("%d-%m-%Y %H:%M:%S")
    plt.figtext(0.02, 0.01, f"Generated on: {current_time}", 
            fontsize=10, style='italic', color='black', ha='left')
    
    plt.show()
    
    # Menentukan kategori pencahayaan berdasarkan luminance
    if avg_luminance < 50:
        return f"Cahaya sangat rendah (Night Vision)"
    elif 50 <= avg_luminance < 150:
        return f"Cahaya sedang"
    else:
        return f"Cahaya terang"

 # Load model CNN yang sudah dilatih
model = load_model('cnn_model.h5')
 # Load label kelas
class_labels = list(train_generator.class_indices.keys())

In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Mode Night Vision (Gray + Colormap JET)
    night_vision = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    night_vision = cv2.applyColorMap(night_vision, cv2.COLORMAP_JET)

    # Preprocessing gambar
    img = cv2.resize(frame, (150, 150))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)

    # Prediksi kelas
    pred = model.predict(img)
    label = class_labels[np.argmax(pred)]

    # Analisis spektrum cahaya
    light_status = analyze_light_spectrum(frame)

    # Menampilkan waktu secara real-time
    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Menampilkan hasil di frame
    cv2.putText(frame, f'Class: {label}', (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, light_status, (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(frame, f'Time: {current_time}', (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    cv2.imshow('Frame', frame)
    cv2.imshow('Night Vision', night_vision)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()